In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models

DATA_CANDIDATES = [
    Path("data/tabular/komdigi_hoaks.csv"),
    Path("../../data/tabular/komdigi_hoaks.csv"),
    Path("../../../data/tabular/komdigi_hoaks.csv"),
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("File data/tabular/komdigi_hoaks.csv tidak ditemukan.")

RANDOM_STATE = 42

In [ ]:
# Muat teks artikel dan label topik dari dataset Komdigi
data = pd.read_csv(DATA_PATH, usecols=["title", "body_text", "topics"])
data["text"] = data["title"].fillna("") + " " + data["body_text"].fillna("")
data["topics"] = data["topics"].fillna("").str.strip()
data = data.loc[data["text"].str.strip().ne("") & data["topics"].ne("")].copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(data["topics"])
class_ids = np.arange(len(label_encoder.classes_))

# Pisahkan data secara stratifikasi agar proporsi topik tetap terjaga.
X_train_val_text, X_test_text, y_train_val, y_test = train_test_split(
    data["text"], y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train_text, X_val_text, y_train, y_val = train_test_split(
    X_train_val_text, y_train_val, test_size=0.125,
    random_state=RANDOM_STATE, stratify=y_train_val
)

# TF-IDF hanya di-fit pada data latih untuk mencegah kebocoran data.
vectorizer = TfidfVectorizer(max_features=512, ngram_range=(1, 2), min_df=2)
X_train = vectorizer.fit_transform(X_train_text).toarray().astype(np.float32)
X_val = vectorizer.transform(X_val_text).toarray().astype(np.float32)
X_test = vectorizer.transform(X_test_text).toarray().astype(np.float32)

class_weight_values = compute_class_weight(
    class_weight="balanced", classes=class_ids, y=y_train
)
class_weights = dict(zip(class_ids, class_weight_values))

print(f"Jumlah data: {len(data)}")
print(f"Training: {len(X_train)} | Validasi: {len(X_val)} | Testing: {len(X_test)}")
print(f"Jumlah fitur TF-IDF: {X_train.shape[1]}")

In [ ]:
# Distribusi topik dan indeks label untuk model
print("Kelas:", dict(enumerate(label_encoder.classes_)))
print("Distribusi data:")
print(data["topics"].value_counts().to_string())

In [ ]:
# CNN gambar diganti dengan jaringan dense untuk fitur teks TF-IDF
tf.keras.utils.set_random_seed(RANDOM_STATE)
model = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dense(len(label_encoder.classes_), activation="softmax")
])

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val, y_val),
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
# Grafik akurasi dan loss selama pelatihan
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="Training")
axes[0].plot(history.history["val_accuracy"], label="Validasi")
axes[0].set_title("Akurasi")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Training")
axes[1].plot(history.history["val_loss"], label="Validasi")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Evaluasi akhir pada data tes yang tidak digunakan saat pelatihan
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")
print(f"Macro-F1      : {macro_f1:.4f}")
print("\nLaporan klasifikasi per topik:")
print(classification_report(
    y_test,
    y_pred,
    labels=class_ids,
    target_names=label_encoder.classes_,
    zero_division=0
))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Contoh hasil prediksi salah satu artikel pada data tes
sample_index = 0
actual_topic = label_encoder.inverse_transform([y_test[sample_index]])[0]
predicted_topic = label_encoder.inverse_transform([y_pred[sample_index]])[0]
print("Judul artikel:", X_test_text.iloc[sample_index][:200])
print("Topik aktual  :", actual_topic)
print("Topik prediksi:", predicted_topic)